## Import library

In [1]:
import pandas as pd
import numpy as np
import os
import json
import time
import joblib
from pathlib import Path
from typing import List, Dict, Any, Optional
from tqdm.auto import tqdm

# PDF Readers
from pypdf import PdfReader

# Langhchain framework
from langchain_core.documents import Document
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# HF + Groq clients
from huggingface_hub import InferenceClient
from groq import Groq


import warnings
warnings.filterwarnings("ignore")

## Load API keys for credential key

In [2]:
from dotenv import load_dotenv

ENV_PATH = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=ENV_PATH)

HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(f"HF token loaded:  {HUGGINGFACE_API_KEY[:3]}...")
print(f"GROQ token loaded: {GROQ_API_KEY[:3]}...")

HF token loaded:  hf_...
GROQ token loaded: gsk...


## Load database

In [3]:
# Load dataset
df = pd.read_parquet("../credit_risk_production/database/data/merged_credit_risk_data.parquet")

# Load feature importance data
fi = pd.read_parquet("../credit_risk_production/database/data/features_data.parquet")

pd.set_option('display.max_columns', None)
print(f"Dataset & Feature Importance shape: {df.shape} & {fi.shape}")
df.head(3)

Dataset & Feature Importance shape: (51336, 87) & (51336, 47)


,PROSPECTID,Total_TL,Tot_Closed_TL,Tot_Active_TL,Total_TL_opened_L6M,Tot_TL_closed_L6M,pct_tl_open_L6M,pct_tl_closed_L6M,pct_active_tl,pct_closed_tl,Total_TL_opened_L12M,Tot_TL_closed_L12M,pct_tl_open_L12M,pct_tl_closed_L12M,Tot_Missed_Pmnt,Auto_TL,CC_TL,Consumer_TL,Gold_TL,Home_TL,PL_TL,Secured_TL,Unsecured_TL,Other_TL,Age_Oldest_TL,Age_Newest_TL,time_since_recent_payment,time_since_first_deliquency,time_since_recent_deliquency,num_times_delinquent,max_delinquency_level,max_recent_level_of_deliq,num_deliq_6mts,num_deliq_12mts,num_deliq_6_12mts,max_deliq_6mts,max_deliq_12mts,num_times_30p_dpd,num_times_60p_dpd,num_std,num_std_6mts,num_std_12mts,num_sub,num_sub_6mts,num_sub_12mts,num_dbt,num_dbt_6mts,num_dbt_12mts,num_lss,num_lss_6mts,num_lss_12mts,recent_level_of_deliq,tot_enq,CC_enq,CC_enq_L6m,CC_enq_L12m,PL_enq,PL_enq_L6m,PL_enq_L12m,time_since_recent_enq,enq_L12m,enq_L6m,enq_L3m,MARITALSTATUS,EDUCATION,AGE,GENDER,NETMONTHLYINCOME,Time_With_Curr_Empr,pct_of_active_TLs_ever,pct_opened_TLs_L6m_of_L12m,pct_currentBal_all_TL,CC_utilization,CC_Flag,PL_utilization,PL_Flag,pct_PL_enq_L6m_of_L12m,pct_CC_enq_L6m_of_L12m,pct_PL_enq_L6m_of_ever,pct_CC_enq_L6m_of_ever,max_unsec_exposure_inPct,HL_Flag,GL_Flag,last_prod_enq2,first_prod_enq2,Credit_Score,Approved_Flag
0,1,5,4,1,0,0,0.000,0.0,0.2,0.8,0,0,0.00,0.0,0,0,0,0,1,0,4,1,4,0,72,18,549,35,15,11,29,29,0,0,0,-99999,-99999,0,0,21,5,11,0,0,0,0,0,0,0,0,0,29,6,0,0,0,6,0,0,566,0,0,0,Married,12TH,48,M,51000,114,0.2,0.0,0.798,-99999.0,0,0.798,1,0.0,0.0,0.0,0.0,13.333,1,0,PL,PL,696,P2
1,2,1,0,1,0,0,0.000,0.0,1.0,0.0,1,0,1.00,0.0,0,0,0,1,0,0,0,0,1,0,7,7,47,-99999,-99999,0,-99999,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,209,1,0,0,Single,GRADUATE,23,F,19000,50,1.0,0.0,0.370,-99999.0,0,-99999.000,0,0.0,0.0,0.0,0.0,0.860,0,0,ConsumerLoan,ConsumerLoan,685,P2
2,3,8,0,8,1,0,0.125,0.0,1.0,0.0,2,0,0.25,0.0,1,1,0,6,1,0,0,2,6,0,47,2,302,11,3,9,25,25,1,9,8,25,25,0,0,10,5,10,0,0,0,0,0,0,0,0,0,25,4,0,0,0,0,0,0,587,0,0,0,Married,SSC,40,M,18,191,1.0,0.5,0.585,-99999.0,0,-99999.000,0,0.0,0.0,0.0,0.0,5741.667,1,0,ConsumerLoan,others,693,P2


## Load ML Models

In [4]:
MODEL_DIR = Path("../credit_risk_production/models/credit_risk")
MODEL_SUBDIR = MODEL_DIR / "ml_credit_risk"

ml_models = {
    model_file.stem: joblib.load(model_file)
    for model_file in MODEL_SUBDIR.glob("*.joblib")
}
model_bundle = joblib.load(MODEL_DIR / "model_bundle.joblib")
parameters = json.loads((MODEL_DIR / "params_credit_risk" / "best_parameters.json").read_text())
metadata = json.loads((MODEL_DIR / "metadata_credit_risk" / "metadata.json").read_text())
metrics_df = pd.read_csv(MODEL_DIR / "metrics_credit_risk" / "model_metrics.csv")

print(f"Loaded ML models: {list(ml_models.keys())}")
print(f"Loaded best parameters: {parameters}")
print(f"Loaded metadata: {metadata}")
print(f"Loaded model bundle: {model_bundle}")
print(metrics_df)

Loaded ML models: ['k_nearest_neighbors', 'logistic_regression', 'gradient_boosting', 'random_forest', 'xgboost', 'decision_tree']
Loaded best parameters: {'Logistic Regression': {'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 2000, 'C': 100}, 'Random Forest': {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 30}, 'Gradient Boosting': {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 3, 'learning_rate': 0.01}, 'XGBoost': {'subsample': 0.9, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 1.0}, 'K-Nearest Neighbors': {'weights': 'distance', 'n_neighbors': 9, 'metric': 'manhattan'}, 'Decision Tree': {'min_samples_split': 5, 'min_samples_leaf': 4, 'max_depth': 10}}
Loaded metadata: {'model_files': {'Logistic Regression': 'logistic_regression.joblib', 'Random Forest': 'random_forest.joblib', 'Gradient Boosting': 'gradient_boosting.joblib', 'XGBoost': 'xgboost.joblib', 'K-Nearest Neighbors': 'k_

## Identify the actual trained estimator inside the bundle


In [5]:
def pick_estimator(bundle):
    if not isinstance(bundle, dict):
        return bundle

    for key in ['models', 'scaler', 'label_encoders', 'feature_columns', 'class_labels']:
        if key in bundle:
            return bundle[key]

    # Fallback: first object that has predict
    for v in bundle.values():
        if hasattr(v, 'predict'):
            return v

    raise ValueError("No estimator found in the bundle.")

risk_model = pick_estimator(model_bundle)
print("✅ Risk model:", type(risk_model).__name__)

✅ Risk model: dict


## Defin one best model or model chosen

In [23]:
risk_model = model_bundle['models']['Gradient Boosting']
scaler = model_bundle['scaler']
label_encoders = model_bundle['label_encoders']

print(f"Selected Model: {type(risk_model).__name__}")

Selected Model: GradientBoostingClassifier


## Identify feature columns used by the model

In [ ]:
if isinstance(model_bundle, dict) and "feature_columns" in model_bundle:
    MODEL_FEATURES = model_bundle["feature_columns"]
elif "feature_columns" in metadata:
    MODEL_FEATURES = metadata["feature_columns"]
else:
    raise ValueError("Feature names not found in model bundle or metadata.")

print(f"Number of model features: {len(MODEL_FEATURES)}")
print(f"Model features: {MODEL_FEATURES}")

Number of model features: 47
Model features: ['num_dbt_12mts', 'pct_tl_open_L12M', 'num_lss_12mts', 'num_lss_6mts', 'pct_tl_closed_L12M', 'pct_tl_closed_L6M', 'time_since_first_deliquency', 'MARITALSTATUS', 'time_since_recent_deliquency', 'first_prod_enq2', 'PL_Flag', 'tot_enq', 'GENDER', 'num_sub', 'last_prod_enq2', 'num_times_60p_dpd', 'Tot_TL_closed_L6M', 'pct_opened_TLs_L6m_of_L12m', 'GL_Flag', 'max_deliq_6mts', 'pct_currentBal_all_TL', 'num_sub_12mts', 'EDUCATION', 'max_recent_level_of_deliq', 'max_delinquency_level', 'pct_tl_open_L6M', 'num_lss', 'CC_Flag', 'num_times_delinquent', 'num_sub_6mts', 'max_unsec_exposure_inPct', 'HL_Flag', 'Tot_TL_closed_L12M', 'Credit_Score', 'PL_utilization', 'num_dbt_6mts', 'CC_utilization', 'num_std', 'Time_With_Curr_Empr', 'Total_TL_opened_L6M', 'time_since_recent_payment', 'Tot_Missed_Pmnt', 'NETMONTHLYINCOME', 'recent_level_of_deliq', 'num_dbt', 'AGE', 'max_deliq_12mts']


## Scale score for one customer

In [40]:
TARGET = "Approved_Flag"

def score_customer(row: pd.Series) -> Dict[str, Any]:
    """Return ML risk probability for a single customer row."""
    df_input = pd.DataFrame([row[MODEL_FEATURES].to_dict()])

    # Apply LableEncoders to categorical features
    for col, le in label_encoders.items():
        if col in df_input.columns:
            # Handle unseen categories labels
            val = str(df_input[col].iloc[0])
            if val in le.classes_:
                df_input[col] = le.transform([val])
            else:
                df_input[col] = 0 # Fallback index for unseen categories

    # Apply StandardScaler
    X_scaled = scaler.transform(df_input[MODEL_FEATURES])

    # Predict class probabilities and label
    proba = risk_model.predict_proba(X_scaled)[0]
    predicted_class = int(risk_model.predict(X_scaled)[0])

    # Map probability per class label [0, 1, 2, 3]
    class_probabilities = {
        f"class_{cls}_prob": round(float(prob), 4)
        for cls, prob in zip(risk_model.classes_, proba)
    }

    return {
        "model_used": type(risk_model).__name__,
        "predicted_flag": predicted_class,
        "class_probabilities": class_probabilities,
        "primary_risk_probability": round(float(proba[predicted_class]), 4)
    }

# Usage
sample = df.iloc[0]
print(f"Score on customer: {score_customer(sample)}")

Score on customer: {'model_used': 'GradientBoostingClassifier', 'predicted_flag': 1, 'class_probabilities': {'class_0_prob': 0.0283, 'class_1_prob': 0.9073, 'class_2_prob': 0.0358, 'class_3_prob': 0.0286}, 'primary_risk_probability': 0.9073}


# 📚 Document Ingestion — PDF → Chunks (per-document strategy)

## Read all PDFs

In [43]:
PDF_DIR = Path("../credit_risk_production/database/pdf")

pdf_files = {
    "delinquency":  PDF_DIR / "Delinquency_Classification .pdf",
    "fraud":        PDF_DIR / "Fraud_Typologies_and_Red Flags .pdf",
    "regulatory":   PDF_DIR / "Regulatory_Risk Policy Core .pdf",
    "scorecard":    PDF_DIR / "Scorecard_Cut-off Policy.pdf"
}

raw_texts: Dict[str, str] = {}

for name, path in pdf_files.items():
    reader = PdfReader(str(path))
    text = "\n".join((page.extract_text() or "") for page in reader.pages)
    raw_texts[name] = text
    print(f"{name:12s} | pages={len(reader.pages):3d} | chars={len(text):,}")

delinquency  | pages= 93 | chars=191,238
fraud        | pages=  9 | chars=25,113
regulatory   | pages= 11 | chars=34,419
scorecard    | pages=178 | chars=465,508


## Chunking strategy to retrieve Document augmentation for each PDFs to enrich vocab on LLM
- #### Strategy 1: Delinquency Classification — rule-based chunking
- #### Strategy 2: Fraud Typologies — semantic paragraph chunking
- #### Strategy 3: Regulatory Policy — recursive with heading preservation
- #### Strategy 4: Scorecard Cut-off — table-aware chunking
- #### Final Strategy: Combine all chunks

#### Strategy 1: Delinquency Classification — rule-based chunking

In [59]:
def chunk_delinquency(text: str) -> List[Document]:
    """Split by classification headings, falls back to paragraph split."""
    keywords = ["Standard", "Sub-standard", "Substandrad", "Doubtful", "Loss"]

    # Crude split: find the index of each keyword and slice
    positions = []
    for kw in keywords:
        idx = text.lower().find(kw.lower())
        if idx != -1:
            positions.append((idx, kw))
    positions.sort()

    docs: List[Document] = []
    if not positions:
        splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
        for i, chunk in enumerate(splitter.split_text(text)):
            docs.append(Document(
                page_content=chunk,
                metadata={"doc_type": "delinquency", "chunk_id": i, "class_level": "unknown"}
            ))
        return docs

    for i, (start, kw) in enumerate(positions):
        end = positions[i + 1][0] if i + 1 < len(positions) else len(text)
        chunk = text[start:end].strip()

        if len(chunk) < 30:
            continue

        docs.append(Document(
            page_content=chunk,
            metadata={"doc_type": "delinquency", "class_level": kw.lower(), "chunk_id": i},
        ))

    return docs

# Usage on delinquency PDF
deling_docs = chunk_delinquency(raw_texts["delinquency"])
print(f"Delinquency chunks: {len(deling_docs)}")
print(deling_docs[3].page_content[:300], "\n--")

Delinquency chunks: 4
sub-standard' immediately on restructuring, 
all borrowers, with the exception of the borrowal categories specified in para 14.1 below ( i.e 
consumer and personal advances, advances classified as capital market and real estate 
exposures), will be entitled to retain the asset classification upon re 
--


### Strategy 2: Fraud Typologies — semantic paragraph chunking

In [60]:
def chunk_fraud(text: str) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=700, 
        chunk_overlap=80,
        separators=["\n\n\n", "\n\n", "\n", ". ", " "]
    )
    docs = []
    for i, chunk in enumerate(splitter.split_text(text)):
        docs.append(Document(
            page_content=chunk,
            metadata={"doc_type": "fraud", "chunk_id": i, "typology": "unspecified"}
        ))
    return docs

# Usage on fraud PDF
fraud_docs = chunk_fraud(raw_texts["fraud"])
print(f"Fraud chunks: {len(fraud_docs)}")
print(fraud_docs[1].page_content[:300], "\n--")

Fraud chunks: 41
[Integrating AI, Real-Time Alerts, and Compliance Automation] 
 
Thirupurasundari Chandrasekaran¹ 
Sr Project Manager, Phoenix, USA¹ 
 
ABSTRACT: Financial institutions are facing unprecedented levels of fraud risk driven by increasing digital 
transaction volumes, rapid adoption of mobile banking,  
--


### Strategy 3: Regulatory Policy — recursive with heading preservation

In [73]:
def chunk_regulatory(text: str) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800, 
        chunk_overlap=120,
        separators=["\n\n", "\n", ". ", " "]
    )
    docs = []
    for i, chunk in enumerate(splitter.split_text(text)):
        docs.append(Document(
            page_content=chunk,
            metadata={"doc_type": "regulatory", "chunk_id": i},
        ))
    return docs

# Usage on regulatory PDF
regulatory_docs = chunk_regulatory(raw_texts["regulatory"])
print(f"Regulatory chunks: {len(regulatory_docs)}")
print(regulatory_docs[1].page_content[:411], "\n--")

Regulatory chunks: 53
(FPC). However, despite these guidelines, rising consumer complaints indicate a gap between regulatory 
expectations and actual practices. This study aims to empirically examine the compliance of FPC norms among 
Banks and Housing Finance Companies (HFCs), as perceived by lending officials and borrowers. Primary data 
were collected from 294 borrowers and 102 lending branches using structured questionnaires. 
--


### Strategy 4: Scorecard Cut-off — table-aware chunking

In [82]:
import re

def chunk_scorecard(text: str) -> List[Document]:
    """Detect numeric ranges like 700-750 or 700 - 750 and split accordingly."""
    pattern = re.compile(r"(\d{3})\s*[--to]+\s*(\d{3})")
    matches = list(pattern.finditer(text))

    docs: List[Document] = []
    if not matches:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=700, 
            chunk_overlap=80,
            separators=["\n\n", "\n", ". ", " "]
        )
        for i, chunk in enumerate(splitter.split_text(text)):
            docs.append(Document(
                page_content=chunk,
                metadata={"doc_type": "scorecard", "chunk_id": i, "min_score": None, "max_score": None}
            ))
        return docs

    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        chunk = text[start:end].strip()

        if len(chunk) < 30:
            continue
        docs.append(Document(
            page_content=chunk,
            metadata={
                "doc_type": "scorecard",
                "chunk_id": i,
                "min_score": int(m.group(1)),
                "max_score": int(m.group(2))
            }
        ))
    return docs

# Usage on scorecard PDF
score_docs = chunk_scorecard(raw_texts["scorecard"])
print(f"Scorecard chunks: {len(score_docs)}")
print(score_docs[1].page_content[:300], "\n--")

Scorecard chunks: 12
300 to 850.  
 
Credit bureau scores consider five general groups of predictive variables: 
– Previous performance, including the severity and frequency of poor performance and 
how recently the poor performance occurred. 
– Current level and use of nonmortgage debt. 
– Amount of time that credit ha 
--


### Combine all chunks

In [83]:
all_docs: List[Document] = deling_docs + fraud_docs + regulatory_docs + score_docs
print(f"Total chunks: {len(all_docs)}")
print(f"By type: {pd.Series([d.metadata['doc_type'] for d in all_docs]).value_counts().to_dict()}")

Total chunks: 110
By type: {'regulatory': 53, 'fraud': 41, 'scorecard': 12, 'delinquency': 4}


## 🧩 Customer-Row → Narrative Document

## Build feature groups

In [84]:
FEATURE_GROUPS = {
    "trade_lines": [
        "Total_TL", "Tot_Closed_TL", "Tot_Active_TL",
        "Total_TL_opened_L6M", "Tot_TL_closed_L6M",
        "pct_tl_open_L6M", "pct_tl_closed_L6M",
        "pct_active_tl", "pct_closed_tl",
        "Total_TL_opened_L12M", "Tot_TL_closed_L12M",
        "pct_tl_open_L12M", "pct_tl_closed_L12M",
    ],
    "product_mix": [
        "Auto_TL", "CC_TL", "Consumer_TL", "Gold_TL",
        "Home_TL", "PL_TL", "Secured_TL", "Unsecured_TL", "Other_TL",
    ],
    "payment_behavior": ["Tot_Missed_Pmnt", "Age_Oldest_TL", "Age_Newest_TL"],
    "delinquency": [
        "time_since_recent_payment", "time_since_first_deliquency",
        "time_since_recent_deliquency", "num_times_delinquent",
        "max_delinquency_level", "max_recent_level_of_deliq",
        "num_deliq_6mts", "num_deliq_12mts", "num_deliq_6_12mts",
        "max_deliq_6mts", "max_deliq_12mts",
        "num_times_30p_dpd", "num_times_60p_dpd",
        "num_std", "num_std_6mts", "num_std_12mts",
        "num_sub", "num_sub_6mts", "num_sub_12mts",
        "num_dbt", "num_dbt_6mts", "num_dbt_12mts",
        "num_lss", "num_lss_6mts", "num_lss_12mts",
        "recent_level_of_deliq",
    ],
    "enquiries": [
        "tot_enq", "CC_enq", "CC_enq_L6m", "CC_enq_L12m",
        "PL_enq", "PL_enq_L6m", "PL_enq_L12m",
        "time_since_recent_enq", "enq_L12m", "enq_L6m", "enq_L3m",
    ],
    "demographics": ["MARITALSTATUS"],
}

## Derived features

In [88]:
def add_derived_features(row: pd.Series) -> pd.Series:
    """Add derived features to a row based on feature groups."""
    r = row.copy()
    def safe_div(a, b): return float(a) / float(b) if pd.notna(a) and pd.notna(b) and b not in (0, None) else 0.0

    r["delinq_velocity"]   = safe_div(row.get("num_deliq_6mts", 0), (row.get("num_deliq_12mts", 0) or 0) + 1)
    r["enq_velocity"]      = safe_div(row.get("enq_L3m", 0), (row.get("enq_L12m", 0) or 0) + 1)
    r["unsecured_ratio"]   = safe_div(row.get("Unsecured_TL", 0), (row.get("Total_TL", 0) or 0) + 1)
    r["active_ratio"]      = safe_div(row.get("Tot_Active_TL", 0), (row.get("Total_TL", 0) or 0) + 1)
    r["recent_open_ratio"] = safe_div(row.get("Total_TL_opened_L6M", 0), (row.get("Total_TL", 0) or 0) + 1)
    r["dpd_score"]         = float(row.get("num_times_30p_dpd", 0) or 0) * 1 + float(row.get("num_times_60p_dpd", 0) or 0) * 2
    r["asset_class_score"] = (
        float(row.get("num_sub", 0) or 0) * 1
        + float(row.get("num_dbt", 0) or 0) * 2
        + float(row.get("num_lss", 0) or 0) * 3
    )
    return r

df_enriched = df.apply(add_derived_features, axis=1)
print("Added derived features")
df_enriched[["delinq_velocity","enq_velocity","unsecured_ratio","dpd_score","asset_class_score"]].head()

Added derived features


,delinq_velocity,enq_velocity,unsecured_ratio,dpd_score,asset_class_score
0,0.0,0.00000,0.666667,0.0,0.0
1,0.0,0.00000,0.500000,0.0,0.0
2,0.1,0.00000,0.666667,0.0,0.0
3,0.0,1.00001,0.500000,0.0,0.0
4,0.0,0.00000,0.000000,0.0,0.0


## Convert each row to a narrative text chunk

In [93]:
def row_to_narrative(row: pd.Series, customer_id: Any = None) -> str:
    """Convert a customer row to a narrative string."""
    parts = [f"CUSTOMER_PROFILE id={customer_id}"]

    for group, cols in FEATURE_GROUPS.items():
        lines = [f" {c}={row[c]}" for c in cols if c in row.index and pd.notna(row[c])]
        if lines:
            parts.append(f"[{group.upper()}]\n" + "\n".join(lines))

    derived_cols = ["delinq_velocity", "enq_velocity", "unsecured_ratio",
                    "active_ratio", "recent_open_ratio", "dpd_score", "asset_class_score"]
    dl = [f" {c}={row[c]:.4f}" for c in derived_cols if c in row.index and pd.notna(row[c])]
    if dl:
        parts.append("[DERIVED]\n" + "\n".join(dl))

    return "\n".join(parts)

# id column: use index if no explicit id column
id_col = "customer_id" if "customer_id" in df_enriched.columns else None

# Build function customer doc
def make_customer_doc(row: pd.Series) -> Document:
    cid = row[id_col] if id_col else row.name
    return Document(
        page_content=row_to_narrative(row, cid),
        metadata={"doc_type": "customer_profile", "customer_id": str(cid)}
    )

# Check usage
print(make_customer_doc(df_enriched.iloc[0]).page_content[:600])

CUSTOMER_PROFILE id=0
[TRADE_LINES]
 Total_TL=5
 Tot_Closed_TL=4
 Tot_Active_TL=1
 Total_TL_opened_L6M=0
 Tot_TL_closed_L6M=0
 pct_tl_open_L6M=0.0
 pct_tl_closed_L6M=0.0
 pct_active_tl=0.2
 pct_closed_tl=0.8
 Total_TL_opened_L12M=0
 Tot_TL_closed_L12M=0
 pct_tl_open_L12M=0.0
 pct_tl_closed_L12M=0.0
[PRODUCT_MIX]
 Auto_TL=0
 CC_TL=0
 Consumer_TL=0
 Gold_TL=1
 Home_TL=0
 PL_TL=4
 Secured_TL=1
 Unsecured_TL=4
 Other_TL=0
[PAYMENT_BEHAVIOR]
 Tot_Missed_Pmnt=0
 Age_Oldest_TL=72
 Age_Newest_TL=18
[DELINQUENCY]
 time_since_recent_payment=549
 time_since_first_deliquency=35
 time_since_recent_deliquen


In [101]:
# Build customer documents sample
SAMPLE_N = 2000
df_sample = df_enriched.sample(min(SAMPLE_N, len(df_enriched)), random_state=42)

customer_docs = [make_customer_doc(r) for _, r in tqdm(df_sample.iterrows(), total=len(df_sample))]
print(f"Customer docs sample: {len(customer_docs)}")
print("Customer doc sample:")
print(customer_docs[:1])

  0%|          | 0/2000 [00:00<?, ?it/s]

Customer docs sample: 2000
Customer doc sample:
[Document(metadata={'doc_type': 'customer_profile', 'customer_id': '8564'}, page_content='CUSTOMER_PROFILE id=8564\n[TRADE_LINES]\n Total_TL=3\n Tot_Closed_TL=1\n Tot_Active_TL=2\n Total_TL_opened_L6M=0\n Tot_TL_closed_L6M=0\n pct_tl_open_L6M=0.0\n pct_tl_closed_L6M=0.0\n pct_active_tl=0.667\n pct_closed_tl=0.333\n Total_TL_opened_L12M=0\n Tot_TL_closed_L12M=0\n pct_tl_open_L12M=0.0\n pct_tl_closed_L12M=0.0\n[PRODUCT_MIX]\n Auto_TL=1\n CC_TL=0\n Consumer_TL=0\n Gold_TL=0\n Home_TL=0\n PL_TL=0\n Secured_TL=2\n Unsecured_TL=1\n Other_TL=2\n[PAYMENT_BEHAVIOR]\n Tot_Missed_Pmnt=0\n Age_Oldest_TL=51\n Age_Newest_TL=30\n[DELINQUENCY]\n time_since_recent_payment=138\n time_since_first_deliquency=14\n time_since_recent_deliquency=12\n num_times_delinquent=2\n max_delinquency_level=26\n max_recent_level_of_deliq=26\n num_deliq_6mts=0\n num_deliq_12mts=0\n num_deliq_6_12mts=0\n max_deliq_6mts=0\n max_deliq_12mts=0\n num_times_30p_dpd=0\n num_times_

## 🗂️ Vector Store (ChromaDB + HuggingFace embeddings)

In [102]:
# Embedding model
EMBED_MODEL = "all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    encode_kwargs={"normalize_embeddings": True}
)
print(f"✅ Embedding model is ready: {EMBED_MODEL}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model is ready: all-MiniLM-L6-v2


## Persist collection to disk

In [107]:
PERSIST_DIR = Path("./chroma_store")
PERSIST_DIR.mkdir(parents=True, exist_ok=True)

def build_or_load_collection(name: str, docs: List[Document]):
    store = Chroma(
        collection_name=name,
        embedding_function=embeddings,
        persist_directory=str(PERSIST_DIR)
    )
    existing = store._collection.count()

    if existing == 0 and docs:
        # add in batches to avoid memory spikes
        BATCH = 256
        for i in range(0, len(docs), BATCH):
            store.add_documents(docs[i:i+BATCH])
        store.persist()
    print(f"Collection '{name}': {store._collection.count()} vectors")
    return store

# 3 Collections (customers + policies)
customer_store = build_or_load_collection("customers", customer_docs)
policy_docs = deling_docs + fraud_docs + regulatory_docs + score_docs
policy_store = build_or_load_collection("policies", policy_docs)

Collection 'customers': 2000 vectors
Collection 'policies': 110 vectors


## 🤖 LLM Client with HF → Groq Fallback

In [ ]:
# Login to Hugging Face Hub
from huggingface_hub import login
login(token=HUGGINGFACE_API_KEY)
print("✅ Logged in to Hugging Face Hub")

✅ Logged in to Hugging Face Hub


In [128]:
# Initialize HF and GROQ clients
hf_client = InferenceClient(api_key=HUGGINGFACE_API_KEY, provider="featherless-ai") if HUGGINGFACE_API_KEY else None
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

def llm_chat(message: List[Dict[str, str]], max_tokens: int = 512, temperature: float = 0.3) -> Dict[str, Any]:
    """
    Try HF first, fallback to Groq.
        Returns a dict with 'text', 'provider'
    """
    if hf_client:
        try:
            resp = hf_client.chat.completions.create(
                model="meta-llama/Llama-3.2-3B-Instruct",
                messages=message,
                max_tokens=max_tokens,
                temperature=temperature
            )
            return {"text": resp.choices[0].message.content,
                    "provider": "huggingface"}
        except Exception as e:
            print(f"❌  Hugging Face failed: {str(e)}")

    # Fallback to Groq
    if groq_client:
        try:
            resp = groq_client.chat.completions.create(
                model="openai/gpt-oss-20b",
                messages=message,
                max_tokens=max_tokens,
                temperature=temperature
            )
            return {"text": resp.choices[0].message.content,
                    "provider": "groq"}
        except Exception as e:
            print(f"❌  Groq failed: {str(e)}")

    raise RuntimeError("⚠️ No LLM provider available. Please check your API keys.")

# Sanity test
print(llm_chat([
    {"role": "user",
     "content": "Reply with the single word: READY"}
]))

{'text': 'READY', 'provider': 'huggingface'}


## 🧠 RAG Pipeline

In [129]:
# Retrieve relevant policy + similar documents
def retrieve_context(row: pd.Series, k_customers: int = 3, k_policies: int = 5) -> Dict[str, Any]:
    """Retrieve relevant policy and similar customer documents for a given row."""
    # Build query text from the customer's narrative + derived features
    query = row_to_narrative(row)
    cust_hits = customer_store.similarity_search(query, k=k_customers)

    # Policy hits: bias query toward delinquency + enquiries since those drive risk
    policy_query = (
        f"{query}\n"
        f"focus: delinquency {row.get('num_deliq_6mts', 0)} misses in 6M"
        f"unsecured ratio {row.get('unsecured_ratio', 0):.2f}"
        f"recent enquiries L3M {row.get('enq_L3m', 0)}"
        f"asset class score {row.get('asset_class_score', 0)}"
    )
    policy_hits = policy_store.similarity_search(policy_query, k=k_policies)

    return {"customers": cust_hits, "policies": policy_hits}

## BUild prompt (system + context + task)

In [ ]:
SYSTEM_PROMPT = """You are a senior credit risk analyst assistant.
You DO NOT invent a risk score. The ML model already produced it.
Your job is to:
    1. Explain the key risk drivers in plain language.
    2. Map the case to the retrieved policies and cite them.
    3. Recommend a concrete action (approve / manual review / decline / reduce limit / flag fraud).
Return STRICT JSON only. No prose outside JSON.
"""

def build_prompt(row: pd.Series, ml_results: Dict[str, Any], context: Dict[str, Any]) -> List[Dict[str, str]]:
    # --- Context blocks ---
    similar_customer_txt = "\n\n".join(
        f"[Similar customer {i+1}]\n{d.page_content[:600]}"
        for i, d in enumerate(context["customers"])
    )
    policies_txt = "\n\n".join(
        f"[Policy ref {i+1} | type={d.metadata.get('doc_type')}]\n{d.page_content}"
        for i, d in enumerate(context["policies"])
        )

    user_prompt = f"""
        CUSTOMER FEATURES:
        {row_to_narrative(row)}

        ML RISK OUTPUT:
        risk_score (0-1): {ml_results['risk_score']:.4f}
        risk_score_100: {ml_results['risk_score_100']:.2f}
        predicted_flag: {ml_results['predicted_flag']}

        SIMILAR HISTORICAL CUSTOMERS:
        {similar_customer_txt}

        RETRIEVED POLICIES:
        {policies_txt}

        TASK:
        Return ONLY a JSON object with EXACTLY these keys:
        {{
            "risk_band": "Low" | "Medium" | "Medium-High" | "High",
            "key_drivers": ["...", "..."],
            "recommended_action": "...",
            "policy_refs": ["...", "..."],
            "fraud_indicators": ["..."],
            "confidence": 0.0-1.0
        }}
        """

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]